In [16]:
import pandas as pd 

data = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")
test_ids = test["PassengerId"]

In [17]:
data

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [18]:
def clean(data):
    
    cols = ["SibSp", "Parch", "Fare", "Age"]
    classes = [1, 2, 3]

    for col in cols:
        for clas in classes:
            median = data.loc[data["Pclass"] == clas, col].median()

            data.loc[
                (data["Pclass"] == clas) & (data[col].isna()),
                col
            ] = median

    data["Embarked"] = data["Embarked"].fillna("U")

    return data

data = clean(data)
test = clean(test)

In [19]:
# feature engineer 
for df in [data,test]:
    df["family_size"] = df["Parch"] + df["SibSp"] + 1
    df["is_child"] = (df["Age"] < 14).astype(int)
    df["is_alone"] = (df["family_size"] == 1).astype(int)
    df["Title"] = df["Name"].str.extract(r",\s*([^.]*)\.")
    

In [20]:
data = data.drop(["Cabin","Name", "PassengerId"],axis=1)
test = test.drop(["Cabin","Name", "PassengerId"],axis=1)


In [21]:
print (data["Ticket"].nunique())

681


In [22]:
print(len(data))

891


In [23]:
ticket_counts = pd.concat([data["Ticket"], test["Ticket"]]).value_counts()
for df in [data, test]:
    df["TicketGroupSize"] = df["Ticket"].map(ticket_counts)

In [24]:
data

,Survived,Pclass,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,family_size,is_child,is_alone,Title,TicketGroupSize
0,0,3,male,22.0,1,0,A/5 21171,7.2500,S,2,0,0,Mr,1
1,1,1,female,38.0,1,0,PC 17599,71.2833,C,2,0,0,Mrs,2
2,1,3,female,26.0,0,0,STON/O2. 3101282,7.9250,S,1,0,1,Miss,1
3,1,1,female,35.0,1,0,113803,53.1000,S,2,0,0,Mrs,2
4,0,3,male,35.0,0,0,373450,8.0500,S,1,0,1,Mr,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,211536,13.0000,S,1,0,1,Rev,1
887,1,1,female,19.0,0,0,112053,30.0000,S,1,0,1,Miss,1
888,0,3,female,24.0,1,2,W./C. 6607,23.4500,S,4,0,0,Miss,4
889,1,1,male,26.0,0,0,111369,30.0000,C,1,0,1,Mr,1


In [25]:
data = data.drop("Ticket" , axis = 1)

In [26]:
agemean = data["Age"].mean()
agestd = data["Age"].std()
faremean = data["Fare"].mean()
farestd = data["Fare"].std()

data["Age"] = (data["Age"]-agemean)/agestd
test["Age"] = (test["Age"] - agemean) / agestd

data["Fare"] = (data["Fare"]-faremean)/farestd
test["Fare"] = (test["Fare"] - faremean) / farestd

In [27]:
data

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,family_size,is_child,is_alone,Title,TicketGroupSize
0,0,3,male,-0.533534,1,0,-0.502163,S,2,0,0,Mr,1
1,1,1,female,0.674512,1,0,0.786404,C,2,0,0,Mrs,2
2,1,3,female,-0.231523,0,0,-0.488580,S,1,0,1,Miss,1
3,1,1,female,0.448003,1,0,0.420494,S,2,0,0,Mrs,2
4,0,3,male,0.448003,0,0,-0.486064,S,1,0,1,Mr,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,-0.156020,0,0,-0.386454,S,1,0,1,Rev,1
887,1,1,female,-0.760043,0,0,-0.044356,S,1,0,1,Miss,1
888,0,3,female,-0.382528,1,2,-0.176164,S,4,0,0,Miss,4
889,1,1,male,-0.231523,0,0,-0.044356,C,1,0,1,Mr,1


In [28]:
data["Sex"] = data["Sex"].map({"male": 0, "female": 1})
test["Sex"] = test["Sex"].map({"male": 0, "female": 1})
data = pd.get_dummies(data, columns=["Embarked","Pclass", "Title"], dtype=int)
test = pd.get_dummies(test, columns=["Embarked","Pclass", "Title"], dtype=int)

test = test.reindex(columns=data.drop("Survived", axis=1).columns, fill_value=0)

KeyError: "None of [Index(['Embarked', 'Pclass', 'Title'], dtype='str')] are in the [columns]"

In [30]:
data

,Survived,Sex,Age,SibSp,Parch,Fare,family_size,is_child,is_alone,TicketGroupSize,...,Title_Master,Title_Miss,Title_Mlle,Title_Mme,Title_Mr,Title_Mrs,Title_Ms,Title_Rev,Title_Sir,Title_the Countess
0,0,NaN,-0.533534,1,0,-0.502163,2,0,0,1,...,0,0,0,0,1,0,0,0,0,0
1,1,NaN,0.674512,1,0,0.786404,2,0,0,2,...,0,0,0,0,0,1,0,0,0,0
2,1,NaN,-0.231523,0,0,-0.488580,1,0,1,1,...,0,1,0,0,0,0,0,0,0,0
3,1,NaN,0.448003,1,0,0.420494,2,0,0,2,...,0,0,0,0,0,1,0,0,0,0
4,0,NaN,0.448003,0,0,-0.486064,1,0,1,1,...,0,0,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,NaN,-0.156020,0,0,-0.386454,1,0,1,1,...,0,0,0,0,0,0,0,1,0,0
887,1,NaN,-0.760043,0,0,-0.044356,1,0,1,1,...,0,1,0,0,0,0,0,0,0,0
888,0,NaN,-0.382528,1,2,-0.176164,4,0,0,4,...,0,1,0,0,0,0,0,0,0,0
889,1,NaN,-0.231523,0,0,-0.044356,1,0,1,1,...,0,0,0,0,1,0,0,0,0,0


In [31]:
x_train = data.drop("Survived", axis = 1)
y_train = data["Survived"]

In [65]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

x_train, x_dev, y_train, y_dev = train_test_split(x_train,y_train, test_size = 0.15, random_state=42)


In [68]:
from sklearn.linear_model import LogisticRegression
import optuna 
from sklearn.model_selection import cross_val_score

def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 50, 200)
    max_depth = trial.suggest_int("max_depth", 2, 10)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
    max_features = trial.suggest_categorical("max_features",["sqrt", "log2", None])

    clf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        random_state=42,
        n_jobs=-1
    )

    score = cross_val_score(clf, x_train, y_train, cv = 5, scoring = "accuracy").mean()

    return score 

study = optuna.create_study(direction="maximize")
study.optimize(objective,n_trials=200)

print(study.best_value)
print(study.best_params)




[I 2026-08-22 22:42:24,424] A new study created in memory with name: no-name-c123d37b-f6a2-48ac-b9c6-c029320392f2
[I 2026-08-22 22:42:25,187] Trial 0 finished with value: 0.8192763157894737 and parameters: {'n_estimators': 164, 'max_depth': 4, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': None}. Best is trial 0 with value: 0.8192763157894737.
[I 2026-08-22 22:42:26,010] Trial 1 finished with value: 0.8129605263157893 and parameters: {'n_estimators': 176, 'max_depth': 9, 'min_samples_split': 3, 'min_samples_leaf': 3, 'max_features': 'log2'}. Best is trial 0 with value: 0.8192763157894737.
[I 2026-08-22 22:42:26,469] Trial 2 finished with value: 0.8234649122807017 and parameters: {'n_estimators': 91, 'max_depth': 5, 'min_samples_split': 9, 'min_samples_leaf': 9, 'max_features': 'log2'}. Best is trial 2 with value: 0.8234649122807017.
[I 2026-08-22 22:42:26,957] Trial 3 finished with value: 0.8192543859649122 and parameters: {'n_estimators': 93, 'max_depth': 10, 'min_samp

0.8318859649122807
{'n_estimators': 153, 'max_depth': 5, 'min_samples_split': 4, 'min_samples_leaf': 1, 'max_features': None}


In [69]:
best_clf = RandomForestClassifier(
    **study.best_params,
    random_state=42,
    n_jobs=-1
)

best_clf.fit(x_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",153
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",4
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",None
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total numb

In [70]:
accuracy = best_clf.score(x_dev,y_dev)
print(accuracy)

0.8214285714285714


In [ ]:
submission_preds = best_clf.predict(test)

In [71]:


df = pd.DataFrame({"PassengerId":test_ids.values, "Survived" : submission_preds })

df.to_csv("submissionforest.csv", index = False)